# Notebook 15: Rank the candidate genes transparently

**Goal.** Combine seven independent evidence layers into one ranking (rank product) with a family-wise permutation p-value, replacing the arbitrary 'bridge score' of the first guide.

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** Builds one table with a row per candidate gene and seven columns, each from an earlier notebook: **abs_delta** (loss of coupling), **abs_meta_z** (replicated aging change, only if all cohorts agree), **tf_target** (target of a regulator in the network), **magma_z** (human Alzheimer's gene-level Z), **pp_h4** (colocalisation), **pert_concord** (agrees with CCCP), **human_concord** (agrees with human aging). Missing evidence is filled with 0 (no support), never with a guess.

*You should see:* Count, mean and max for 7 columns over 63 genes (some columns are constant in the simulation, which is harmless: ties share ranks).

In [ ]:
from mlc.genetics import load_orthologs
from mlc.analysis import get_net, rank_product
from mlc import plots
SIM = RAW / "sim"
orth = load_orthologs(SIM / "HOM_MouseHumanSequence_sim.rpt")
h = orth.drop_duplicates("Symbol_mouse").set_index("Symbol_mouse").Symbol_human
rew = pd.read_csv(RES / "rewiring_genes.csv").set_index("gene")
sig = pd.read_csv(RES / "age_signature_meta.csv", index_col=0)
gz = pd.read_csv(SIM / "ad_gene_level_sim.csv").set_index("SYMBOL").ZSTAT
col = pd.read_csv(RES / "coloc_results.csv").set_index("gene")["PP.H4"]
conc = pd.read_csv(RES / "perturb_concordance.csv", index_col=0).pert_concord
hga = pd.read_csv(RES / "human_gene_age.csv").set_index("gene").beta_age
E = pd.DataFrame(index=rew.index)
E["abs_delta"] = rew.delta.abs()
E["abs_meta_z"] = sig.meta_z.abs().reindex(E.index) * sig.concordant.reindex(E.index).astype(float)
E["tf_target"] = E.index.isin(get_net("mouse").target).astype(float)
E["magma_z"] = h.reindex(E.index).map(gz)
E["pp_h4"] = col.reindex(E.index)
E["pert_concord"] = conc.reindex(E.index)
hs = np.sign(h.reindex(E.index).map(hga)); E["human_concord"] = (hs == np.sign(sig.meta_z.reindex(E.index))).astype(float)
E = E.fillna(0); E.to_csv(RES / "evidence_table.csv"); print(E.describe().round(2).loc[["count", "mean", "max"]])

**Cell 3.** **Rank product.** Within each layer genes are ranked (1 = strongest). A gene's score is the geometric mean of its seven ranks, so a gene must be good in *several* layers, not brilliant in one. The permutation p shuffles ranks independently within each layer 5,000 times and asks how often the best gene by chance would score as well as yours (**family-wise**, so it is already corrected for testing all genes).

*You should see:* Ctsd first (family-wise p about 0.003), then Bnip3 (about 0.08), Ctsb, Gba, Pink1...: only the gene with support from many layers (including colocalisation) is significant after correcting for all genes. This is the honest result.

In [ ]:
cols = ["abs_delta", "abs_meta_z", "tf_target", "magma_z", "pp_h4", "pert_concord", "human_concord"]
ranked = rank_product(E, cols, n_perm=5000, seed=SEED)
ranked.to_csv(RES / "candidate_ranking.csv"); print(ranked.head(10).round(3)[["rank_product", "p_familywise"] + cols])

**Cell 4.** **Figure 6c-d.** (a) The top genes by rank product. (b) Which evidence layers support each gene (darker = stronger rank). Pick 3-5 genes for experiments *before* reading their literature, then check novelty.

In [ ]:
plots.fig_candidates(ranked); plt.show()